In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("../dane/merged_data.csv", encoding="utf-8")

for col in ["TMAX", "TMIN", "STD", "TMNG", "SMDB", "PKSN"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for status_col, value_col in {
    "WTMAX": "TMAX",
    "WTMIN": "TMIN",
    "WSTD": "STD",
    "WTMNG": "TMNG",
    "WSMDB": "SMDB",
    "WPKSN": "PKSN"
}.items():
    df[status_col] = df[status_col].astype("string").fillna("")
    df.loc[df[status_col] == "8", value_col] = np.nan

df["DATE"] = pd.to_datetime(
    df[["ROK", "MC", "DZ"]].rename(columns={"ROK": "year", "MC": "month", "DZ": "day"}),
    errors="coerce"
)

df = df.sort_values(["NSP", "DATE"]).reset_index(drop=True)

df["TMAX_target"] = df.groupby("NSP")["TMAX"].shift(-1)

for lag in [1, 2, 3]:
    df[f"TMAX_lag_{lag}"] = df.groupby("NSP")["TMAX"].shift(lag)
    df[f"TMIN_lag_{lag}"] = df.groupby("NSP")["TMIN"].shift(lag)
    df[f"STD_lag_{lag}"] = df.groupby("NSP")["STD"].shift(lag)
    df[f"SMDB_lag_{lag}"] = df.groupby("NSP")["SMDB"].shift(lag)

df["month"] = df["DATE"].dt.month
df["day_of_year"] = df["DATE"].dt.dayofyear

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import pandas as pd

features = [
    "TMAX", "TMIN", "STD", "TMNG", "SMDB", "PKSN",
    "TMAX_lag_1", "TMAX_lag_2", "TMAX_lag_3",
    "TMIN_lag_1", "TMIN_lag_2", "TMIN_lag_3",
    "STD_lag_1", "STD_lag_2", "STD_lag_3",
    "SMDB_lag_1", "SMDB_lag_2", "SMDB_lag_3",
    "month", "day_of_year"
]

# df = pd.read_csv("../dane/merged_data.csv", parse_dates=["DATE"])

model_df = df[features + ["TMAX_target", "DATE"]].dropna().copy()

split_date = model_df["DATE"].quantile(0.8)
train = model_df[model_df["DATE"] <= split_date]
test = model_df[model_df["DATE"] > split_date]

X_train = train[features]
y_train = train["TMAX_target"]
X_test = test[features]
y_test = test["TMAX_target"]

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = root_mean_squared_error(y_test, pred)

print("MAE:", mae)
print("RMSE:", rmse)

ValueError: Missing column provided to 'parse_dates': 'DATE'